# Лабораторная работа №1 (практика №2), АЗСИИ
## Атаки FGSM и DeepFool на нейронные сети: оценка стойкости моделей

**Цель работы:** на готовом проекте [EEL6812_DeepFool_Project](https://github.com/ewatson2/EEL6812_DeepFool_Project)
разобраться, как устроены две состязательные (adversarial) атаки — **FGSM** и **DeepFool**. Нужно оценить стойкость
четырёх обученных моделей (FC-500-150 и LeNet на MNIST, Network-in-Network и LeNet на CIFAR-10)
и выяснить, как параметр `eps` атаки FGSM влияет на результат.

Ноутбук рассчитан на **Google Colab** (Среда выполнения → Сменить среду выполнения → T4 GPU).
Если GPU нет, он запустится и на CPU, только медленнее (DeepFool займёт десятки минут).

Нумерация разделов совпадает с пунктами задания.

## 1. Клонирование проекта

In [ ]:
import os

if not os.path.isdir('EEL6812_DeepFool_Project'):
    !git clone https://github.com/ewatson2/EEL6812_DeepFool_Project

## 2. Переход в папку проекта

In [ ]:
%cd EEL6812_DeepFool_Project

### Исправление совместимости `deepfool` с CPU

В `deepfool()` устройство определяется так: `device = images.get_device()`. На GPU это возвращает номер видеокарты (0),
и всё работает. Но в современных версиях PyTorch на CPU `get_device()` возвращает **-1**, и следующая строка
`torch.arange(...).to(-1)` падает с ошибкой `RuntimeError: Device index must not be negative`. Заменяем одну строку
на `images.device`: так работает и на GPU, и на CPU. На результаты атаки это не влияет.

In [ ]:
p = 'utils/project_utils.py'
src = open(p, encoding='utf-8').read()
if 'device = images.get_device()' in src:
    open(p, 'w', encoding='utf-8').write(src.replace('device = images.get_device()', 'device = images.device'))
    print('project_utils.py: get_device() -> .device')

## 3. Импорт библиотек

In [ ]:
import numpy as np
import json, torch
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, models
from torchvision.transforms import transforms

## 4. Импорт вспомогательных модулей проекта

* `models/project_models.py` — архитектуры сетей (`FC_500_150`, `LeNet_CIFAR`, `LeNet_MNIST`, `Net` = Network-in-Network);
* `utils/project_utils.py` — реализация атак `fgsm`, `deepfool` и обёртки `evaluate_attack` (оценка на всём тестовом наборе)
  и `display_attack` (визуализация).

In [ ]:
from models.project_models import FC_500_150, LeNet_CIFAR, LeNet_MNIST, Net
from utils.project_utils import get_clip_bounds, evaluate_attack, display_attack

## 5–6. Фиксация генератора случайных чисел

`rand_seed` — порядковый номер студента в списке группы (Гугл-таблица). **Замените значение на свой номер.**
От сида зависят разбиение `random_split`, порядок батчей и то, какие картинки случайно выберет `display_attack`.

In [ ]:
rand_seed = 21  # порядковый номер в Гугл-таблице группы

np.random.seed(rand_seed)
torch.manual_seed(rand_seed)

## 7. Выбор устройства (GPU T4 в Colab)

In [ ]:
use_cuda = torch.cuda.is_available()
device = torch.device('cuda' if use_cuda else 'cpu')
print('Устройство:', device, '|', torch.cuda.get_device_name(0) if use_cuda else 'CPU')

## 8. Загрузка датасета MNIST

Картинки нормализуются: `x_norm = (x - mean) / std`. При `mean = std = 0.5` пиксели из диапазона [0, 1] переходят в [-1, 1].
`get_clip_bounds` возвращает границы допустимых значений **в нормализованном пространстве**, то есть -1 и 1.
Атаки обрезают (clamp) состязательное изображение по этим границам, чтобы оно осталось «настоящей» картинкой.

In [ ]:
mnist_mean = 0.5
mnist_std = 0.5
mnist_dim = 28

mnist_min, mnist_max = get_clip_bounds(mnist_mean, mnist_std, mnist_dim)
mnist_min = mnist_min.to(device)
mnist_max = mnist_max.to(device)

mnist_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mnist_mean, std=mnist_std)])

mnist_tf_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=mnist_mean, std=mnist_std)])

mnist_tf_inv = transforms.Compose([
    transforms.Normalize(mean=0.0, std=np.divide(1.0, mnist_std)),
    transforms.Normalize(mean=np.multiply(-1.0, mnist_std), std=1.0)])

mnist_temp = datasets.MNIST(root='datasets/mnist', train=True, download=True, transform=mnist_tf_train)
mnist_train, mnist_val = random_split(mnist_temp, [50000, 10000])

mnist_test = datasets.MNIST(root='datasets/mnist', train=False, download=True, transform=mnist_tf)

print('MNIST: train', len(mnist_train), '| val', len(mnist_val), '| test', len(mnist_test))
print('Границы клиппинга MNIST:', mnist_min.min().item(), mnist_max.max().item())

## 9. Загрузка датасета CIFAR-10

In [ ]:
cifar_mean = [0.491, 0.482, 0.447]
cifar_std = [0.202, 0.199, 0.201]
cifar_dim = 32

cifar_min, cifar_max = get_clip_bounds(cifar_mean, cifar_std, cifar_dim)
cifar_min = cifar_min.to(device)
cifar_max = cifar_max.to(device)

cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)])

cifar_tf_train = transforms.Compose([
    transforms.RandomCrop(size=cifar_dim, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=cifar_mean, std=cifar_std)])

cifar_tf_inv = transforms.Compose([
    transforms.Normalize(mean=[0.0, 0.0, 0.0], std=np.divide(1.0, cifar_std)),
    transforms.Normalize(mean=np.multiply(-1.0, cifar_mean), std=[1.0, 1.0, 1.0])])

cifar_temp = datasets.CIFAR10(root='datasets/cifar-10', train=True, download=True, transform=cifar_tf_train)
cifar_train, cifar_val = random_split(cifar_temp, [40000, 10000])

cifar_test = datasets.CIFAR10(root='datasets/cifar-10', train=False, download=True, transform=cifar_tf)

cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

print('CIFAR-10: train', len(cifar_train), '| val', len(cifar_val), '| test', len(cifar_test))
print('Границы клиппинга CIFAR-10 по каналам:',
      [round(v, 3) for v in cifar_min[:, 0, 0].tolist()], [round(v, 3) for v in cifar_max[:, 0, 0].tolist()])

## 10. DataLoader'ы и параметры DeepFool

В тексте задания переменная `deep_args` используется, но не определяется. Берём её из оригинального ноутбука проекта
(`Model_Demo_Adv.ipynb`); значения такие же, как в статье DeepFool:
`[batch_size, num_classes, overshoot, max_iters] = [10, 10, 0.02, 50]`.

In [ ]:
batch_size = 64
workers = 4

mnist_loader_train = DataLoader(mnist_train, batch_size=batch_size, shuffle=True, num_workers=workers)
mnist_loader_val = DataLoader(mnist_val, batch_size=batch_size, shuffle=False, num_workers=workers)
mnist_loader_test = DataLoader(mnist_test, batch_size=batch_size, shuffle=False, num_workers=workers)

cifar_loader_train = DataLoader(cifar_train, batch_size=batch_size, shuffle=True, num_workers=workers)
cifar_loader_val = DataLoader(cifar_val, batch_size=batch_size, shuffle=False, num_workers=workers)
cifar_loader_test = DataLoader(cifar_test, batch_size=batch_size, shuffle=False, num_workers=workers)

deep_batch_size = 10    # сколько логитов (классов) обрабатывать за один проход
deep_num_classes = 10   # число классов, среди которых ищем ближайшую границу
deep_overshoot = 0.02   # «перелёт» через границу решения, чтобы точно сменить класс
deep_max_iters = 50     # максимум итераций на одно изображение

deep_args = [deep_batch_size, deep_num_classes, deep_overshoot, deep_max_iters]

### Важный момент: готовые результаты в папке `results/`

В репозитории уже лежат CSV-файлы с результатами авторов (`results/cifar_nin_fgsm.csv` и т.д.).
`evaluate_attack` устроена так: если CSV с таким именем уже есть, атака **не пересчитывается**, функция просто читает
файл и печатает метрики (это сделано, чтобы долгий расчёт можно было продолжить после обрыва).
Без следующей ячейки мы получили бы чужие цифры. Поэтому переносим авторские результаты в `results_author/`
и считаем атаки заново.

In [ ]:
import shutil

if os.path.isdir('results') and not os.path.isdir('results_author'):
    shutil.move('results', 'results_author')
os.makedirs('results', exist_ok=True)
print('Авторские результаты:', sorted(os.listdir('results_author')))

## 11. Network-in-Network (CIFAR-10): стойкость к FGSM и DeepFool

Метрики, которые печатает `evaluate_attack`:
* **Test Error** — доля тестовых картинок, которые после атаки классифицируются неверно (чем выше, тем сильнее атака);
* **Robustness** — средняя относительная величина возмущения `‖r‖₂ / ‖x‖₂` (насколько сильно пришлось изменить картинку).
  Для DeepFool это оценка стойкости модели ρ_adv из статьи: чем меньше значение, тем легче обмануть сеть;
* **Time** — время работы атаки.

In [ ]:
fgsm_eps = 0.2
model = Net().to(device)
model.load_state_dict(torch.load('weights/clean/cifar_nin.pth', map_location=torch.device('cpu')))

evaluate_attack('cifar_nin_fgsm.csv', 'results', device, model, cifar_loader_test,
                cifar_min, cifar_max, fgsm_eps, is_fgsm=True)
print('')
evaluate_attack('cifar_nin_deepfool.csv', 'results', device, model, cifar_loader_test,
                cifar_min, cifar_max, deep_args, is_fgsm=False)

if device.type == 'cuda':
    torch.cuda.empty_cache()

## 12. LeNet (CIFAR-10): стойкость к FGSM и DeepFool

In [ ]:
fgsm_eps = 0.1
model = LeNet_CIFAR().to(device)
model.load_state_dict(torch.load('weights/clean/cifar_lenet.pth', map_location=torch.device('cpu')))

evaluate_attack('cifar_lenet_fgsm.csv', 'results', device, model, cifar_loader_test,
                cifar_min, cifar_max, fgsm_eps, is_fgsm=True)
print('')
evaluate_attack('cifar_lenet_deepfool.csv', 'results', device, model, cifar_loader_test,
                cifar_min, cifar_max, deep_args, is_fgsm=False)

if device.type == 'cuda':
    torch.cuda.empty_cache()

## 13. Визуализация состязательных примеров

`display_attack` берёт `fig_rows` случайных картинок из тестового набора и выводит для каждой пять колонок:
оригинал, результат FGSM, результат DeepFool, возмущение FGSM и возмущение DeepFool.
Под картинками подписаны предсказанный класс и `conf score`. Это **логит** (сырое значение выхода сети),
а не вероятность, поэтому он может быть больше 1.

> В задании для пункта 13 `torch.load` вызывается без `map_location`. Если веса сохранены на GPU, а сессия работает
> на CPU, такой вызов упадёт, поэтому ниже везде указан `map_location=device`.

### LeNet (MNIST)

In [ ]:
fgsm_eps = 0.6
model = LeNet_MNIST().to(device)
model.load_state_dict(torch.load('weights/clean/mnist_lenet.pth', map_location=device))

display_attack(device, model, mnist_test, mnist_tf_inv, mnist_min, mnist_max, fgsm_eps, deep_args,
               has_labels=False, l2_norm=True, pert_scale=1.0, fig_rows=2, fig_width=25, fig_height=11)

if device.type == 'cuda':
    torch.cuda.empty_cache()

### FC-500-150 (MNIST)

In [ ]:
fgsm_eps = 0.2
model = FC_500_150().to(device)
model.load_state_dict(torch.load('weights/clean/mnist_fc.pth', map_location=device))

display_attack(device, model, mnist_test, mnist_tf_inv, mnist_min, mnist_max, fgsm_eps, deep_args,
               has_labels=False, l2_norm=True, pert_scale=1.0, fig_rows=2, fig_width=25, fig_height=11)

if device.type == 'cuda':
    torch.cuda.empty_cache()

### Network-in-Network (CIFAR-10)

In [ ]:
fgsm_eps = 0.2
model = Net().to(device)
model.load_state_dict(torch.load('weights/clean/cifar_nin.pth', map_location=device))

display_attack(device, model, cifar_test, cifar_tf_inv, cifar_min, cifar_max, fgsm_eps, deep_args,
               has_labels=False, l2_norm=True, pert_scale=1.0, fig_rows=2, fig_width=25, fig_height=11,
               label_map=cifar_classes)

if device.type == 'cuda':
    torch.cuda.empty_cache()

### LeNet (CIFAR-10)

In [ ]:
fgsm_eps = 0.1
model = LeNet_CIFAR().to(device)
model.load_state_dict(torch.load('weights/clean/cifar_lenet.pth', map_location=device))

display_attack(device, model, cifar_test, cifar_tf_inv, cifar_min, cifar_max, fgsm_eps, deep_args,
               has_labels=False, l2_norm=True, pert_scale=1.0, fig_rows=2, fig_width=25, fig_height=11,
               label_map=cifar_classes)

if device.type == 'cuda':
    torch.cuda.empty_cache()

### Для сравнения: DeepFool на MNIST

Пункты 11–12 оценивают только CIFAR-10. Чтобы в отчёте было с чем сравнивать модели MNIST, прогоняем такую же
оценку для FC-500-150 и LeNet на MNIST (с теми же `fgsm_eps`, что и в пункте 13).

In [ ]:
for name, cls, w, eps in [('mnist_lenet', LeNet_MNIST, 'weights/clean/mnist_lenet.pth', 0.6),
                          ('mnist_fc', FC_500_150, 'weights/clean/mnist_fc.pth', 0.2)]:
    model = cls().to(device)
    model.load_state_dict(torch.load(w, map_location=device))
    print('=====', name, '| fgsm_eps =', eps)
    evaluate_attack(name + '_fgsm.csv', 'results', device, model, mnist_loader_test,
                    mnist_min, mnist_max, eps, is_fgsm=True)
    print('')
    evaluate_attack(name + '_deepfool.csv', 'results', device, model, mnist_loader_test,
                    mnist_min, mnist_max, deep_args, is_fgsm=False)
    print('')

if device.type == 'cuda':
    torch.cuda.empty_cache()

## 14. Исследование влияния `fgsm_eps` (0.001, 0.02, 0.5, 0.9, 10)

FGSM строит состязательный пример за **один шаг**:

$$x_{adv} = \mathrm{clip}\big(x + \varepsilon \cdot \mathrm{sign}(\nabla_x L(f(x), y)),\ x_{min},\ x_{max}\big)$$

Каждый пиксель сдвигается ровно на ±ε в сторону роста ошибки, после чего картинка обрезается до допустимого диапазона.
**Важно:** ε задаётся в *нормализованном* пространстве. В пикселях [0, 1] сдвиг равен `ε·std`:
* MNIST (`std = 0.5`): ε = 0.02 соответствует 0.01 яркости (≈ 2.5/255), а диапазон значений [-1, 1] имеет ширину 2;
* CIFAR-10 (`std ≈ 0.2`): ε = 0.02 соответствует ≈ 0.004 (≈ 1/255), а ширина диапазона ≈ 5.

Для каждой из четырёх моделей и каждого ε считаем на всём тестовом наборе (10 000 картинок):
* `test_error_%` — ошибку после атаки (как в `evaluate_attack`);
* `robustness` — ‖r‖₂/‖x‖₂ (как в `evaluate_attack`);
* `pixel_change` — средний модуль фактического изменения пикселя в шкале [0, 1] (после клиппинга);
* `top_class_%` — долю самого частого предсказанного класса. Если она близка к 100%, сеть отвечает на все картинки
  одним и тем же классом.

`evaluate_attack` только печатает метрики и ничего не возвращает. Поэтому для таблицы используется своя функция
`fgsm_eval`. Внутри она вызывает ту же функцию `fgsm` из проекта и ту же `compute_robustness`.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from utils.project_utils import fgsm, compute_robustness

eps_list = [0.001, 0.02, 0.5, 0.9, 10]

experiments = {
    'FC-500-150 / MNIST': dict(cls=FC_500_150, weights='weights/clean/mnist_fc.pth', dataset=mnist_test,
                               loader=mnist_loader_test, cmin=mnist_min, cmax=mnist_max,
                               std=mnist_std, inv=mnist_tf_inv, labels=None),
    'LeNet / MNIST': dict(cls=LeNet_MNIST, weights='weights/clean/mnist_lenet.pth', dataset=mnist_test,
                          loader=mnist_loader_test, cmin=mnist_min, cmax=mnist_max,
                          std=mnist_std, inv=mnist_tf_inv, labels=None),
    'NiN / CIFAR-10': dict(cls=Net, weights='weights/clean/cifar_nin.pth', dataset=cifar_test,
                           loader=cifar_loader_test, cmin=cifar_min, cmax=cifar_max,
                           std=cifar_std, inv=cifar_tf_inv, labels=cifar_classes),
    'LeNet / CIFAR-10': dict(cls=LeNet_CIFAR, weights='weights/clean/cifar_lenet.pth', dataset=cifar_test,
                             loader=cifar_loader_test, cmin=cifar_min, cmax=cifar_max,
                             std=cifar_std, inv=cifar_tf_inv, labels=cifar_classes),
}


def load_model(cfg):
    model = cfg['cls']().to(device)
    model.load_state_dict(torch.load(cfg['weights'], map_location=device))
    model.eval()
    return model


def clean_error(model, loader):
    """Ошибка на чистых (неатакованных) тестовых картинках, в процентах."""
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            correct += (model(images).argmax(dim=1) == labels).sum().item()
    return 100.0 * (1.0 - correct / len(loader.dataset))


def fgsm_eval(model, loader, cmin, cmax, eps, std):
    """FGSM на всём тестовом наборе; возвращает словарь метрик."""
    std_t = torch.tensor(std, dtype=torch.float32, device=device).view(-1, 1, 1)
    correct, p_adv, pix, preds = 0, [], [], []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        _, perts, _, labels_adv = fgsm(model, cmin, cmax, images, labels, eps)
        correct += (labels_adv == labels).sum().item()
        p_adv.append(compute_robustness(images, perts))
        pix.append((perts.abs() * std_t).flatten(start_dim=1).mean(dim=1).detach().cpu().numpy())
        preds.append(labels_adv.cpu().numpy())
    preds = np.concatenate(preds)
    n = len(loader.dataset)
    return {'test_error_%': 100.0 * (1.0 - correct / n),
            'robustness': float(np.mean(np.concatenate(p_adv))),
            'pixel_change': float(np.mean(np.concatenate(pix))),
            'top_class_%': 100.0 * np.bincount(preds, minlength=10).max() / n}

**Проверка.** Убедимся, что `fgsm_eval` считает так же, как `evaluate_attack`: для FC-500-150 при ε = 0.2 ошибка
должна совпасть с результатом из раздела «DeepFool на MNIST» выше.

In [ ]:
_m = load_model(experiments['FC-500-150 / MNIST'])
print('fgsm_eval, FC-500-150, eps=0.2 ->',
      round(fgsm_eval(_m, mnist_loader_test, mnist_min, mnist_max, 0.2, mnist_std)['test_error_%'], 2), '%')

In [ ]:
rows = []
t0 = time.time()
for name, cfg in experiments.items():
    model = load_model(cfg)
    rows.append({'model': name, 'eps': 0.0, 'test_error_%': clean_error(model, cfg['loader']),
                 'robustness': 0.0, 'pixel_change': 0.0, 'top_class_%': np.nan})
    for eps in eps_list:
        r = fgsm_eval(model, cfg['loader'], cfg['cmin'], cfg['cmax'], eps, cfg['std'])
        rows.append({'model': name, 'eps': eps, **r})
    if device.type == 'cuda':
        torch.cuda.empty_cache()

eps_df = pd.DataFrame(rows)
eps_df.to_csv('results/fgsm_eps_study.csv', index=False)
print('Готово за {:.1f} c'.format(time.time() - t0))

### Таблица: ошибка после атаки FGSM, % (`eps = 0` — чистые данные, без атаки)

In [ ]:
err_table = eps_df.pivot(index='model', columns='eps', values='test_error_%').loc[list(experiments)]
err_table.columns = ['clean' if c == 0 else 'eps={:g}'.format(c) for c in err_table.columns]
err_table.round(2)

### Таблица: относительная величина возмущения ‖r‖₂/‖x‖₂ и средний сдвиг пикселя (шкала [0, 1])

In [ ]:
rob_table = eps_df[eps_df.eps > 0].pivot(index='model', columns='eps', values='robustness').loc[list(experiments)]
pix_table = eps_df[eps_df.eps > 0].pivot(index='model', columns='eps', values='pixel_change').loc[list(experiments)]
top_table = eps_df[eps_df.eps > 0].pivot(index='model', columns='eps', values='top_class_%').loc[list(experiments)]
for t in (rob_table, pix_table, top_table):
    t.columns = ['eps={:g}'.format(c) for c in t.columns]
print('robustness (‖r‖/‖x‖):'); display(rob_table.round(3))
print('Средний |Δпикселя| в шкале [0, 1]:'); display(pix_table.round(3))
print('Доля самого частого предсказанного класса, %:'); display(top_table.round(1))

### Графики зависимости от ε

In [ ]:
# Категориальная палитра (фиксированный порядок), текст и сетка — нейтральные
COLORS = ['#2a78d6', '#eb6834', '#1baf7a', '#eda100']
INK, INK2, GRID = '#0b0b0b', '#52514e', '#e4e3df'


def style_axes(ax, title, ylabel):
    ax.set_xscale('log')
    ax.set_xticks(eps_list)
    ax.set_xticklabels([str(e) for e in eps_list])
    ax.set_xlabel('fgsm_eps (лог. шкала)', color=INK2)
    ax.set_ylabel(ylabel, color=INK2)
    ax.set_title(title, color=INK, fontsize=13, loc='left')
    ax.grid(True, color=GRID, linewidth=0.8)
    ax.tick_params(colors=INK2)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'):
        ax.spines[s].set_color(GRID)


fig, axes = plt.subplots(1, 2, figsize=(15, 5.2), facecolor='white')
for i, name in enumerate(experiments):
    d = eps_df[(eps_df.model == name) & (eps_df.eps > 0)]
    clean = eps_df[(eps_df.model == name) & (eps_df.eps == 0)]['test_error_%'].item()
    axes[0].plot(d.eps, d['test_error_%'], color=COLORS[i], lw=2, marker='o', ms=8, label=name)
    axes[0].axhline(clean, color=COLORS[i], lw=1, ls=':', alpha=0.8)
    axes[1].plot(d.eps, d['pixel_change'], color=COLORS[i], lw=2, marker='o', ms=8, label=name)
style_axes(axes[0], 'Ошибка после FGSM (пунктир — ошибка на чистых данных)', 'test error, %')
axes[0].set_ylim(0, 100)
style_axes(axes[1], 'Средний фактический сдвиг пикселя (после клиппинга)', '|Δ пикселя|, шкала [0, 1]')
axes[0].legend(frameon=False, labelcolor=INK)
plt.tight_layout()
plt.show()

### Как выглядят атакованные картинки при разных ε

Для каждой модели берём одну и ту же тестовую картинку своего датасета и показываем оригинал и результат FGSM
при каждом ε. Под каждой картинкой — предсказание модели.

In [ ]:
def eps_gallery(img_idx_mnist=0, img_idx_cifar=0):
    fig, axes = plt.subplots(len(experiments), len(eps_list) + 1, figsize=(15, 11), facecolor='white')
    for r, (name, cfg) in enumerate(experiments.items()):
        model = load_model(cfg)
        idx = img_idx_mnist if cfg['labels'] is None else img_idx_cifar
        x, y = cfg['dataset'][idx]
        x = x.unsqueeze(0).to(device)
        y = torch.tensor([y], device=device)
        lbl = (lambda k: str(k)) if cfg['labels'] is None else (lambda k: cfg['labels'][k])
        with torch.no_grad():
            p0 = model(x).argmax(dim=1).item()
        cols = [('оригинал', x, p0)]
        for eps in eps_list:
            adv, _, _, p = fgsm(model, cfg['cmin'], cfg['cmax'], x, y, eps)
            cols.append(('eps={:g}'.format(eps), adv, p.item()))
        for c, (title, img, pred) in enumerate(cols):
            ax = axes[r, c]
            img = cfg['inv'](img.detach())[0].clamp(0, 1).cpu()
            if img.shape[0] == 1:
                ax.imshow(img[0], cmap='gray', vmin=0, vmax=1)
            else:
                ax.imshow(img.permute(1, 2, 0))
            ok = pred == y.item()
            ax.set_xticks([]); ax.set_yticks([])
            ax.set_xlabel('pred: {} {}'.format(lbl(pred), '✓' if ok else '✗'), fontsize=11, color=INK)
            if r == 0:
                ax.set_title(title, fontsize=13, color=INK)
        axes[r, 0].set_ylabel(name + '\ntrue: ' + lbl(y.item()), fontsize=11, color=INK)
    plt.tight_layout()
    plt.show()


eps_gallery(img_idx_mnist=0, img_idx_cifar=3)

### Выводы по пункту 14

*(заполняется по результатам выполнения — см. ячейку ниже)*